<a href="https://colab.research.google.com/github/lucianoselimaj/SigExt/blob/MMD/notebooks/three_layer_agentic_refinement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#@title requirements
!pip install transformers datasets accelerate evaluate rouge_score torch jsonlines rake_nltk pytorch_lightning
!pip install rapidfuzz
!pip install boto3
!pip install mistralai
!pip install --upgrade mistralai

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Clone the Repo

In [3]:
# @markdown check the check-box to clone the repo from sorce, <b>otherwise it will be loaded from shared google drive files</b>

clone_repo = False # @param {type:"boolean"}
if clone_repo:
  !git clone https://github.com/lucianoselimaj/SigExt.git
  path = "/SigExt"

else:
  path = "/content/drive/MyDrive/DNLP-storage/SigExt"


# Preparing Data

In [4]:
# @title Download requirements for data preparing

%cd {path}

import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')

/content/drive/MyDrive/DNLP-storage/SigExt


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
# @title Run the `prepare_data.py` script
# @markdown Here you can select the dataset you want to use for the training. </br> If you want to use a new dataset, make sure to also change the `path_to_dataset`. </br> No new file will be downloaded or processed if `path_to_dataset` already exists!!

import os
dataset = "cnn" #@param{type:"string"}
path_to_dataset = "experiments/cnn_dataset/" #@param{type:"string"}
if not os.path.exists(path+"/"+path_to_dataset):
  print('New dataset...')
  !python3 src/prepare_data.py --dataset {dataset} --output_dir {path_to_dataset}
  pass

else:
  print(f"Directory already exist. `{path_to_dataset}` will be used")

Directory already exist. `experiments/cnn_dataset/` will be used


# Train

You can modify path to check-points and outputs to load or save different files.
**To keep track of procedure and training routh, insert paths you create in this text down below, under the `paths` ( DO NOT FORGET THE DESCRIPTION :) thank you in advance.  )**



 **⚠️Even for test runs, Change two paths to avoid overwriting existing results⚠️**


---


**Paths**
*   `experiments/cnn_extractor_model/` Baseline code (dataset: cnn),  no modification. output in `experiments/cnn_dataset_with_keyphrase/`
*   `new_path` add you new path here...





In [6]:

path_to_checkpoint = "experiments/cnn_extractor_model/" #@param{type:"string"}
path_to_output = "experiments/cnn_dataset_with_keyphrase/" #@param{type:"string"}

train_longformer_extractor = False #@param{type:"boolean"}
inference_longformer_extractor = False #@param{type:"boolean"}


In [7]:
#@title Calling `train_longformer_extractor_context.py` to train the longformer
if train_longformer_extractor:
  !python3 src/train_longformer_extractor_context.py \
    --dataset_dir {path_to_dataset} \
    --checkpoint_dir {path_to_checkpoint}

In [8]:
# #@title Calling `inference_longformer_extractor.py`
if inference_longformer_extractor:
  !python3 src/inference_longformer_extractor.py \
    --dataset_dir {path_to_dataset} \
    --checkpoint_dir {path_to_checkpoint} \
    --output_dir {path_to_output}

In [9]:
summerizing_result_path = "experiments/extention2_claude/" #@param{type:"string"}
top_k_kp = 15 #@param{type:"integer"}

In [10]:
# CRITIC_PROMPT_TEMPLATE = """<s>[INST]You are a rigorous data auditor. Your objective is to verify a draft summary against a source text with absolute precision.

# Here is the original task, source text, and required key phrases:
# <original_task>
# {original_task}
# </original_task>

# Here is the drafted summary:
# <summary>
# {draft_summary}
# </summary>

# Evaluate the draft for TWO specific failure modes only:
# 1. OMITTED KEY PHRASES: Did the draft fail to include any of the required key phrases?
# 2. FACTUAL DISCREPANCIES: Did the draft include information that contradicts or hallucinates beyond the source text?

# STRICT OUTPUT CONSTRAINTS:
# - If the draft successfully integrates the key phrases and is factually accurate, output exactly the word: "PASS"
# - If there are errors, output ONLY a highly concise, actionable bulleted list of what is missing or incorrect.
# - DO NOT include conversational filler, greetings, explanations, or conclusions.
# - DO NOT critique grammar, tone, style, or sentence length. Focus strictly on missing facts and key phrases.[/INST]"""
# #@title PROMPTS HERE
# ENHANCER_PROMPT_TEMPLATE = """<s>[INST]You are a surgical text editor. Your ONLY job is to insert missing key phrases into a draft summary with zero conversational bloat.

# ### EXAMPLE OF YOUR JOB ###
# Draft: "The company reported a profit for the quarter."
# Critique: "- Missing key phrase: 'Q3'"
# Correct Output: "The company reported a profit for the Q3 quarter."
# Incorrect Output (DO NOT DO THIS): "Here is the updated summary: The company reported a profit for the Q3 quarter."
# Incorrect Output (DO NOT DO THIS): "The company reported a profit for the quarter, specifically Q3."
# ### END EXAMPLE ###

# Here is the original task, source text, and required key phrases:
# <original_task>
# {original_task}
# </original_task>

# Here is the initial draft summary:
# <summary>
# {draft_summary}
# </summary>

# Here is the Critique Report:
# <critique>
# {critique}
# </critique>

# STRICT EDITORIAL RULES:
# 1. DO NOT rewrite sentences.
# 2. DO NOT output conversational filler (e.g., "Here is...", "Updated:").
# 3. SURGICALLY INSERT the missing keywords from the critique into the existing draft.
# 4. If the Critique Report says "PASS", output the draft summary exactly as it was provided.

# Output ONLY the final summary text.[/INST]"""


MISTRAL_CRITIC_PROMPT_TEMPLATE = """<s>[INST]You are a rigorous data auditor. Your objective is to verify a draft summary against a source text with absolute precision.

<original_task>
{original_task}
</original_task>

<summary>
{draft_summary}
</summary>

Evaluate the draft for TWO specific failure modes only:
1. OMITTED KEY PHRASES: Did the draft fail to include the core concept of the required key phrases?
2. FACTUAL DISCREPANCIES: Did the draft include hallucinated information that contradicts the source?

STRICT OUTPUT CONSTRAINTS:
- If the draft successfully captures the key phrases and is factually accurate, output exactly: PASS
- If there are missing key phrases, output ONLY a bulleted list: - MISSING KEY PHRASE: [phrase]
- Do not explain why the phrase is missing.
- Do not critique grammar, style, or length.[/INST]"""

MISTRAL_ENHANCER_PROMPT_TEMPLATE = """<s>[INST]You are an expert journalistic copy editor. Your task is to weave missing information into a news summary naturally and elegantly.

STRICT EDITORIAL RULES:
1. NATURAL EMPHASIS: Do not awkwardly jam or force the exact key phrase into the sentence if it ruins the flow. Instead, adapt the phrasing to naturally emphasize the missing concept.
2. SWAP AND CONDENSE: To keep the sentence short, replace generic or low-value filler words with the missing information.
3. PRESERVE CORE FACTS: Do not delete the main actions or subjects of the original draft.
4. NO NEW SENTENCES: Do not write new explanatory sentences. Work entirely by upgrading the existing sentence structure.
5. PURE TEXT: Output only the final summary text. No introductions, explanations, or formatting tags.

<original_task>
{original_task}
</original_task>

<summary>
{draft_summary}
</summary>

<critique>
{critique}
</critique>

Output:[/INST]"""

# ==============================================================================
# 3. PROMPTS (CLAUDE)
# ==============================================================================
CLAUDE_CRITIC_PROMPT_TEMPLATE = """You are a rigorous data auditor. Your objective is to verify a draft summary against a source text with absolute precision.

<original_task>
{original_task}
</original_task>

<summary>
{draft_summary}
</summary>

Evaluate the draft for TWO specific failure modes only:
1. OMITTED KEY PHRASES: Did the draft fail to include the core concept of the required key phrases?
2. FACTUAL DISCREPANCIES: Did the draft include hallucinated information that contradicts the source?

STRICT OUTPUT CONSTRAINTS:
- If the draft successfully captures the key phrases and is factually accurate, output exactly: PASS
- If there are missing key phrases, output ONLY a bulleted list: - MISSING KEY PHRASE: [phrase]
- Do not explain why the phrase is missing.
- Do not critique grammar, style, or length."""

CLAUDE_ENHANCER_PROMPT_TEMPLATE = """You are an expert journalistic copy editor. Your task is to weave missing information into a news summary naturally and elegantly.

STRICT EDITORIAL RULES:
1. NATURAL EMPHASIS: Do not awkwardly jam or force the exact key phrase into the sentence if it ruins the flow. Instead, adapt the phrasing to naturally emphasize the missing concept.
2. SWAP AND CONDENSE: To keep the sentence short, replace generic or low-value filler words with the missing information.
3. PRESERVE CORE FACTS: Do not delete the main actions or subjects of the original draft.
4. NO NEW SENTENCES: Do not write new explanatory sentences. Work entirely by upgrading the existing sentence structure.
5. PURE TEXT: Output only the final summary text. No introductions, explanations, or formatting tags.

<original_task>
{original_task}
</original_task>

<summary>
{draft_summary}
</summary>

<critique>
{critique}
</critique>

Output:"""

In [11]:
import sys
import os
import time
import json
import multiprocessing
from types import ModuleType

# Mistral & AWS imports
from mistralai.client import Mistral
import boto3.session as boto3_session
import botocore.config

In [12]:
from google.colab import userdata

# ==============================================================================
# GLOBAL CONFIGURATION (YOUR CONTROL PANEL)
# ==============================================================================
CONFIG = {
    # --- Pipeline Settings ---
    "ACTIVE_MODEL": "claude",   # Options: "mistral" or "claude"


    "IS_3_LAYER_ACTIVE": True,

    # --- Mistral API Settings ---
    "MISTRAL_API_KEY": "YdbLhqEepgXav0x8ruTJiWR20GsgWCrD", # Consider putting this in Colab Secrets too!
    "MISTRAL_MODEL_NAME": "mistral-small-latest",

    # --- AWS Bedrock (Claude) Settings ---
    "AWS_REGION": "us-east-1",
    "CLAUDE_MODEL_ID": "us.anthropic.claude-haiku-4-5-20251001-v1:0",

    # --- SECURE AWS CREDENTIALS ---
    # Pass the NAME of the secret, not the key itself!
    "AWS_ACCESS_KEY_ID": userdata.get('AWS_ACCESS_KEY_ID'),
    "AWS_SECRET_ACCESS_KEY": userdata.get('AWS_SECRET_ACCESS_KEY'),
    "AWS_SESSION_TOKEN": None,

    # --- Dataset & Summarization Arguments ---


    "DATASET": dataset,
    "DATASET_DIR": path_to_output,
    "OUTPUT_DIR": summerizing_result_path,
    "KW_STRATEGY": "sigext_topk",
    "KW_MODEL_TOP_K": str(top_k_kp)
}

In [13]:
import boto3
import json
from google.colab import userdata

def test_bedrock_connection():
    print("1. Fetching credentials from Colab Secrets...")
    ak = userdata.get('AWS_ACCESS_KEY_ID')
    sk = userdata.get('AWS_SECRET_ACCESS_KEY')

    if not ak or not sk:
        print("❌ FAILED: Could not find your keys in Colab Secrets.")
        return

    print("2. Starting AWS Bedrock session...")
    try:
        session = boto3.Session(
            aws_access_key_id=ak,
            aws_secret_access_key=sk,
            region_name="us-west-2"
        )
        bedrock = session.client('bedrock-runtime')
    except Exception as e:
        print(f"❌ FAILED: Could not start AWS session. Error: {str(e)}")
        return

    print("3. Pinging Claude Haiku 4.5...")
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 100,
        "messages": [{"role": "user", "content": "Reply with exactly the words: 'HELLO FROM AWS'"}]
    }

    try:
        response = bedrock.invoke_model(
            modelId="us.anthropic.claude-haiku-4-5-20251001-v1:0",
            contentType="application/json",
            accept="application/json",
            body=json.dumps(body)
        )
        response_body = json.loads(response.get("body").read())
        text = response_body["content"][0]["text"]
        print("\n✅ SUCCESS! Claude says:", text)
    except Exception as e:
        print("\n❌ FAILED. AWS Error details:", str(e))

test_bedrock_connection()

1. Fetching credentials from Colab Secrets...
2. Starting AWS Bedrock session...
3. Pinging Claude Haiku 4.5...

✅ SUCCESS! Claude says: HELLO FROM AWS


In [14]:
def chat_mistral(prompt_text, client):
    chat_response = client.chat.complete(
        model=CONFIG["MISTRAL_MODEL_NAME"],
        messages=[{"role": "user", "content": prompt_text}]
    )
    return chat_response.choices[0].message.content

def chat_claude(prompt_text):
    # Retrieve credentials from config, fallback to None if they are empty strings
    aws_access_key_id = CONFIG.get("AWS_ACCESS_KEY_ID") or None
    aws_secret_access_key = CONFIG.get("AWS_SECRET_ACCESS_KEY") or None
    aws_session_token = CONFIG.get("AWS_SESSION_TOKEN") or None

    # Initialize the session with explicit credentials (if provided)
    current_session = boto3_session.Session(
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=aws_session_token
    )

    bedrock = current_session.client(
        service_name="bedrock-runtime",
        region_name=CONFIG["AWS_REGION"],
        config=botocore.config.Config(
            read_timeout=120, connect_timeout=120, retries={"max_attempts": 5}
        ),
    )

    # 1. Setup the API Template
    api_template = {
        "modelId": CONFIG["CLAUDE_MODEL_ID"],
        "contentType": "application/json",
        "accept": "application/json",
    }

    # 2. Build the Payload using the modern Messages API
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": 1024,
        "temperature": 1.0,

        "messages": [
            {
                "role": "user",
                "content": prompt_text.strip()
            }
        ]
    }
    api_template["body"] = json.dumps(body)

    # 3. Invoke the model and extract the text safely
    try:
        response = bedrock.invoke_model(**api_template)
        response_body = json.loads(response.get("body").read())

        # Safely extract the text depending on the Claude schema
        if "content" in response_body:
            # Modern Claude 3 / 3.5 / 4.5 schema
            return response_body["content"][0]["text"].strip()
        elif "completion" in response_body:
            # Legacy Claude 2 schema (just in case you switch back)
            return response_body["completion"].strip()
        else:
            print(f"\n🚨 UNKNOWN SCHEMA. Raw AWS Response: {json.dumps(response_body)}\n")
            return None

    except Exception as e:
        print(f"\n🚨 AWS BEDROCK ERROR: {str(e)}\n")
        raise e  # Let the error bubble up so the outer try/except loop can retry it
def predict_one_eg_mistral_3_layer(prompt_dict, is_3_layer_active=CONFIG["IS_3_LAYER_ACTIVE"]):
    client = Mistral(api_key=CONFIG["MISTRAL_API_KEY"])
    original_task = prompt_dict["prompt_input"]

    draft_summary = ""
    final_output = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            if is_3_layer_active:
                draft_summary = chat_mistral(original_task, client)
            else:
                final_output = chat_mistral(original_task, client)
            break
        except Exception as e:
            if "429" in str(e):
                print("L1 Rate limit hit! Sleeping for 10s...")
                time.sleep(2)

    if not is_3_layer_active: return final_output or "Error: Layer 1 Failed"
    if not draft_summary: return "Error: Layer 1 Failed"

    critic_prompt = MISTRAL_CRITIC_PROMPT_TEMPLATE.format(original_task=original_task, draft_summary=draft_summary)
    critique = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            critique = chat_mistral(critic_prompt, client)
            break
        except Exception as e:
            if "429" in str(e):
                print("L2 Rate limit hit! Sleeping for 10s...")
                time.sleep(2)

    if not critique: return "Error: Layer 2 Failed"
    if len(critique) < 10 and "PASS" in critique:
        return draft_summary

    enhancer_prompt = MISTRAL_ENHANCER_PROMPT_TEMPLATE.format(original_task=original_task, draft_summary=draft_summary, critique=critique)
    final_output = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            final_output = chat_mistral(enhancer_prompt, client)
            break
        except Exception as e:
            if "429" in str(e):
                print("L3 Rate limit hit! Sleeping for 10s...")
                time.sleep(2)

    return final_output or "Error: Layer 3 Failed"


def predict_one_eg_claude_3_layer(prompt_dict, is_3_layer_active=CONFIG["IS_3_LAYER_ACTIVE"]):
    original_task = prompt_dict["prompt_input"]

    draft_summary = ""
    final_output = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            if is_3_layer_active:
                draft_summary = chat_claude(original_task)

                final_output = chat_claude(original_task)
            break
        except Exception as e:
            # ---> ADD THIS LINE TO SEE THE REAL ERROR <---
            print(f"\n🚨 AWS BEDROCK ERROR: {str(e)}\n")

            if "429" in str(e) or "Throttling" in str(e):
                print("AWS L1 Throttled! Sleeping for 5s...")
                time.sleep(5)

            else:

                break # Stop looping if it's a fatal error!

    if not is_3_layer_active: return final_output or "Error: Layer 1 Failed"
    if not draft_summary: return "Error: Layer 1 Failed"

    critic_prompt = CLAUDE_CRITIC_PROMPT_TEMPLATE.format(original_task=original_task, draft_summary=draft_summary)
    critique = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            critique = chat_claude(critic_prompt)
            break
        except Exception as e:
            if "429" in str(e) or "Throttling" in str(e):
                print("AWS L2 Throttled! Sleeping for 5s...")
                time.sleep(5)

    if not critique: return "Error: Layer 2 Failed"
    if len(critique) < 10 and "PASS" in critique:
        return draft_summary

    enhancer_prompt = CLAUDE_ENHANCER_PROMPT_TEMPLATE.format(original_task=original_task, draft_summary=draft_summary, critique=critique)
    final_output = ""
    for _ in range(5):
        try:
            time.sleep(1.1)
            final_output = chat_claude(enhancer_prompt)
            break
        except Exception as e:
            if "429" in str(e) or "Throttling" in str(e):
                print("AWS L3 Throttled! Sleeping for 5s...")
                time.sleep(5)
    return final_output or "Error: Layer 3 Failed"

# ==============================================================================
# 6. MONKEY PATCHING & EXECUTION
# ==============================================================================
class FakePool:
    def __init__(self, processes=None): pass
    def __enter__(self): return self
    def __exit__(self, *args): pass
    def imap(self, func, iterable):
        return map(func, iterable)

multiprocessing.Pool = FakePool

# Inject both our functions into bedrock_utils
m = ModuleType("bedrock_utils")

# 1. The Mistral function
m.predict_one_eg_mistral = predict_one_eg_mistral_3_layer

# 2. The Claude function (mapped to what the script actually calls)
m.predict_one_eg_claude = predict_one_eg_claude_3_layer

# 3. ---> ADD THIS LINE BACK <---
# This satisfies the hardcoded import statement at the top of zs_summarization.py
m.predict_one_eg_claude_instant = predict_one_eg_claude_3_layer

sys.modules["bedrock_utils"] = m

# Pass configurations dynamically to zs_summarization
sys.argv = [
    "zs_summarization.py",
    "--model_name", CONFIG["ACTIVE_MODEL"],
    "--kw_strategy", CONFIG["KW_STRATEGY"],
    "--kw_model_top_k", CONFIG["KW_MODEL_TOP_K"],
    "--dataset", CONFIG["DATASET"],
    "--dataset_dir", CONFIG["DATASET_DIR"],
    "--output_dir", CONFIG["OUTPUT_DIR"]
]

sys.path.append('/content/drive/MyDrive/DNLP-storage/SigExt/src')
import zs_summarization
import importlib
importlib.reload(zs_summarization)

zs_summarization.main()

  0%|          | 0/500 [00:00<?, ?it/s]

A 64-year-old former Japanese school principal, Yuhei Takashima, is facing criminal charges after Tokyo police arrested him for allegedly photographing obscene acts with a girl aged 13 or 14 and producing pornography in Manila. According to Japanese law enforcement investigating crimes involving minors, Takashima confessed to paying for sex with over 12,000 women, some as young as 14, during repeated visits to the Philippines over more than 25 years, with officers seizing 147,600 photos documenting his activities. Takashima told police he began engaging in this behavior in 1988 while working at a Japanese school in Manila and continued returning to the Philippines on vacations because sex was inexpensive, with his case having been under investigation since 2013.


  0%|          | 0/500 [00:18<?, ?it/s]


KeyboardInterrupt: Interrupted by user

In [ ]:
import json

def compare_rouge_files(baseline_filepath, verified_filepath):
    """Loads two JSON metric files and prints a side-by-side comparison."""

    try:
        # Load the JSON data from both files
        with open(baseline_filepath, 'r') as f1:
            baseline_metrics = json.load(f1)

        with open(verified_filepath, 'r') as f2:
            verified_metrics = json.load(f2)

    except FileNotFoundError as e:
        print(f"Error loading files: {e}")
        return
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON: {e}")
        return

    # Print the formatted comparison table
    print("=" * 65)
    print(f"{'Metric':<30} {'Baseline':>12} {'Verified':>12} {'Diff':>10}")
    print("=" * 65)

    # A helper function to print each row and calculate the difference
    def print_row(metric_name, key):
        base_val = baseline_metrics.get(key, 0)
        ver_val = verified_metrics.get(key, 0)
        diff = ver_val - base_val
        # Format with a + sign for positive differences
        diff_str = f"{diff:>+10.4f}"
        print(f"{metric_name:<30} {base_val:>12.4f} {ver_val:>12.4f} {diff_str}")

    # Map your desired table labels to the JSON keys
    print_row("ROUGE-1 F1", "rouge1f")
    print_row("ROUGE-L F1", "rougeLf")
    print_row("ROUGE-1 Recall", "rouge1r")
    print_row("ROUGE-2 F1", "rouge2f")
    print_row("ROUGE-Lsum F1", "rougeLsumf")
    print("-" * 65)
    print_row("Generation Length", "gen_len")
    print("=" * 65)

baseline_path = "/content/drive/MyDrive/DNLP-storage/SigExt/experiments/cnn_extsig_predictions_mistral-small-2603_k15/test_metrics.json" #@param{type:"string"}
verified_path = "/content/drive/MyDrive/DNLP-storage/SigExt/experiments/extention2_claude/test_metrics.json" #@param{type:"string"}


# Run the comparison!
compare_rouge_files(baseline_path , verified_path)

Metric                             Baseline     Verified       Diff
ROUGE-1 F1                          38.8941       0.2896   -38.6045
ROUGE-L F1                          24.2784       0.2896   -23.9888
ROUGE-1 Recall                      47.2585       0.1583   -47.1002
ROUGE-2 F1                          14.1088       0.0000   -14.1088
ROUGE-Lsum F1                       33.5527       0.2896   -33.2631
-----------------------------------------------------------------
Generation Length                   77.5900       5.0000   -72.5900
